In [74]:
%pip install -q --pre mujoco mediapy matplotlib


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Algorithms

They use RK4, which we implemented within our RK45 system. Our system only used RK4 for error estimation to know when to reduce the timestep in our integrator. Mujoco uses RK4 with a fixed timestep to solve the system, which is faster and less accurate than RK45. However, Mujoco recommends using another integrator called "fast implicit-in-velocity" or `implicitfast`. `implicitfast` is a version of their `implicit` integrator that ignores the centripetal and Coriolis forces to increase the computation speed for a slight reduction in accuracy. The main difference between the two is that `implicitfast` uses midpoint integration, whereas RK4 (and RK45) are multistep integrators. The documentation recommends `implicitfast` as the default integrator because it has similar cost to Euler integration and is more stable. While being more expensive to run, RK4 is better for simulations that conserve energy. (Kaelan)

# Example

> Search the examples provided by the library, as well as ones you might find
> online. Identify an example that you find engaging and make it run on your
> computer. To the best of your ability, explain how it works. It should be
> approximately on the level of your understanding from class
> This should be no more than two paragraphs plus annotated code/xml. Label the
> group member that is the author or the individual paragraphs, and the group
> member that was responsible for conducting the runs.

In [75]:
# Other imports and helper functions
import time
import itertools
import numpy as np
import mujoco
import mediapy as media
import matplotlib.pyplot as plt

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

from IPython.display import clear_output
clear_output()

In [76]:
spin_recoil = """
<mujoco>
  <option cone="elliptic" timestep="1e-4" integrator="implicitfast"/>

  <visual>
    <global elevation="0"/>
  </visual>

  <asset>
    <texture name="grid" type="2d" builtin="checker" rgb1=".1 .2 .3" rgb2=".2 .3 .4" width="300" height="300"/>
    <material name="grid" texture="grid" texrepeat="8 8" reflectance=".2"/>
    <texture name="ball" type="cube" builtin="checker" rgb1=".3 .3 .3" rgb2="1 1 1" width="300" height="300"/>
    <material name="ball" texture="ball" texrepeat="8 8"/>
  </asset>

  <worldbody>
    <geom name="floor" size=".2 .2 .01" type="plane" material="grid"/>
    <geom name="wall" size=".2 .2 .01" type="plane" material="grid" pos=".2 0 .2" zaxis="-1 0 0"/>
    <light pos="0 0 .6"/>
    <light pos="-.6 0 .6" dir="1 0 -1"/>
    <camera name="closeup" pos="0 -.5 0.3" xyaxes="1 0 0 0 1 2"/>
    <body name="ball" pos="-.1 0 .2">
      <freejoint/>
      <geom name="ball" type="sphere" size=".02" material="ball" mass=".001"/>
      <body name="ballast">
        <joint type="ball" stiffness="0.1" damping=".0001"/>
        <geom name="ballast" type="sphere" size=".02" contype="0" conaffinity="0" mass=".01" group="3"/>
      </body>
    </body>
  </worldbody>

  <default>
    <pair solref="-100000 0" solreffriction="0 -10000"/>
  </default>

  <contact>
    <pair geom1="floor" geom2="ball"/>
    <pair geom1="wall" geom2="ball"/>
  </contact>

  <keyframe>
    <key name="throw" qvel="1.3 0 0 0 0 0 0 0 0"/>
  </keyframe>
</mujoco>
"""
model = mujoco.MjModel.from_xml_string(spin_recoil)
data = mujoco.MjData(model)

duration = 7    # (seconds)
framerate = 60  # (Hz)

# Simulate and display video.
frames = []
mujoco.mj_resetDataKeyframe(model, data, 0)  # Reset the state to keyframe 0
with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * framerate:
      renderer.update_scene(data, "closeup")
      pixels = renderer.render()
      frames.append(pixels)

media.show_video(frames, fps=framerate)

This browser does not support the video tag.


# Modified Example

> Based on the example you identified, extend/replace it with something novel.
> Explain your extensions, and the physical insights required to understand how
> they work. 
> In addition to the paragraphs, label the group member(s) that are
> responsible for the various components of the extensions/changes made to the
> example.

In [77]:
# Bare-bones: one ball dropping onto a plane.
# Keep geometry/spin setup fixed and tune contact behavior below.

# Contact tuning knobs
friction_slide = 2.0
friction_torsion = 0.02
friction_roll = 0.002

# More elastic contact: lower damping ratio tends to rebound more.
solref_timeconst = 0.003
solref_dampratio = 0.20

# Contact impedance shaping (higher first two values -> firmer contact response).
solimp0 = 0.95
solimp1 = 0.995
solimp2 = 0.001

drop_ball_xml = f"""
<mujoco>
  <option timestep="5e-4" gravity="0 0 -9.81" integrator="RK4"/>

  <asset>
    <texture name="ball_tex" type="cube" builtin="checker" rgb1="0.15 0.15 0.15" rgb2="1 1 1" width="300" height="300"/>
    <material name="ball_mat" texture="ball_tex" texrepeat="8 8"/>
  </asset>

  <worldbody>
    <geom name="floor" type="plane" size="1 1 .1" friction="{friction_slide} {friction_torsion} {friction_roll}"/>
    <light pos="0.2 -0.3 1.0" dir="0 0 -1"/>
    <camera name="side" pos="0 -0.35 0.32" xyaxes="1 0 0 0 1 2"/>
    <!-- <camera name="follow_ball" mode="trackcom" target="ball" pos="0 -0.65 0.32"/> -->

    <body name="ball" pos="0 0 0.8">
      <freejoint/>
      <geom name="ball" type="sphere" size="0.03" mass="0.05" material="ball_mat"
            friction="{friction_slide} {friction_torsion} {friction_roll}"/>
    </body>
  </worldbody>

  <default>
    <pair solref="{solref_timeconst} {solref_dampratio}" solimp="{solimp0} {solimp1} {solimp2}"/>
  </default>

  <contact>
    <pair geom1="floor" geom2="ball"/>
  </contact>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(drop_ball_xml)
data = mujoco.MjData(model)

duration = 4.0
capture_framerate = 90
playback_fps = 25

mujoco.mj_resetData(model, data)
data.qvel[:] = 0.0

# Initial positions
pos_x = 0.0
pos_y = 0.0
pos_z = 0.8

# Initial velocities
vel_x = 0.0
vel_y = 0.0
vel_z = 0.0

# Initial spin
spin_x = 0.0
spin_y = 0.0
spin_z = 25

# For this free joint setup: qpos = [x, y, z, qw, qx, qy, qz]
data.qpos[0] = pos_x
data.qpos[1] = pos_y
data.qpos[2] = pos_z

# For this free joint setup: qvel = [vx, vy, vz, wx, wy, wz]
data.qvel[0] = vel_x
data.qvel[1] = vel_y
data.qvel[2] = vel_z
data.qvel[3] = spin_x
data.qvel[4] = spin_y
data.qvel[5] = spin_z

mujoco.mj_forward(model, data)

print(f"Initial position: x={pos_x}, y={pos_y}, z={pos_z} m")
print(f"Initial velocity: vx={vel_x}, vy={vel_y}, vz={vel_z} m/s")
print(f"Initial spins: wx={spin_x}, wy={spin_y}, wz={spin_z} rad/s")
print(
  f"Contact params -> friction=({friction_slide}, {friction_torsion}, {friction_roll}), "
  f"solref=({solref_timeconst}, {solref_dampratio}), "
  f"solimp=({solimp0}, {solimp1}, {solimp2})"
)

frames = []
with mujoco.Renderer(model) as renderer:
  while data.time < duration:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * capture_framerate:
      renderer.update_scene(data, "side")
      # renderer.update_scene(data, "follow_ball")
      frames.append(renderer.render())

media.show_video(frames, fps=playback_fps)

Initial position: x=0.0, y=0.0, z=0.8 m
Initial velocity: vx=0.0, vy=0.0, vz=0.0 m/s
Initial spins: wx=0.0, wy=0.0, wz=25 rad/s
Contact params -> friction=(2.0, 0.02, 0.002), solref=(0.003, 0.2), solimp=(0.95, 0.995, 0.001)


This browser does not support the video tag.


The initial intent here was to replicate the phenomenon that if you drop a bouncy/basket/dodge ball and spin it, when it bounces it will reverse spin direction. This breif exploration led me to understand this requires a much more sophisticated simulation that captures deformation of the ball and it's elasticity. I am not sure what the best avenues for doing this would be. I'm imagining you'd have to move beyond a simple 'ball' model and actually model the inflatable sphere as well as the internal gas? Interesting quesiton to pursue. (Stellios)

## Modified Example 2:

In [78]:
# Modified Example 2: two balls stacked vertically as the initial condition.

# Reuse the same contact tuning style from Modified Example 1.
friction_slide = 2.0
friction_torsion = 0.02
friction_roll = 0.002
solref_timeconst = 0.003
solref_dampratio = 0.20
solimp0 = 0.95
solimp1 = 0.995
solimp2 = 0.001

two_ball_xml = f"""
<mujoco>
  <option timestep="5e-4" gravity="0 0 -9.81" integrator="RK4"/>

  <asset>
    <texture name="ball_tex" type="cube" builtin="checker" rgb1="0.15 0.15 0.15" rgb2="1 1 1" width="300" height="300"/>
    <material name="ball_mat" texture="ball_tex" texrepeat="8 8"/>
  </asset>

  <worldbody>
    <geom name="floor" type="plane" size="1 1 .1" friction="{friction_slide} {friction_torsion} {friction_roll}"/>
    <light pos="0.2 -0.3 1.0" dir="0 0 -1"/>
    <camera name="side" pos="0 -0.50 0.45" xyaxes="1 0 0 0 1 2"/>

    <body name="ball_lower" pos="0 0 0.24">
      <freejoint/>
      <geom name="ball_lower_geom" type="sphere" size="0.03" mass="0.05" material="ball_mat"
            friction="{friction_slide} {friction_torsion} {friction_roll}"/>
    </body>

    <body name="ball_upper" pos="0 0 0.30">
      <freejoint/>
      <geom name="ball_upper_geom" type="sphere" size="0.03" mass="0.05" material="ball_mat"
            friction="{friction_slide} {friction_torsion} {friction_roll}"/>
    </body>
  </worldbody>

  <default>
    <pair solref="{solref_timeconst} {solref_dampratio}" solimp="{solimp0} {solimp1} {solimp2}"/>
  </default>

  <contact>
    <pair geom1="floor" geom2="ball_lower_geom"/>
    <pair geom1="floor" geom2="ball_upper_geom"/>
    <pair geom1="ball_lower_geom" geom2="ball_upper_geom"/>
  </contact>
</mujoco>
"""

model2 = mujoco.MjModel.from_xml_string(two_ball_xml)
data2 = mujoco.MjData(model2)

duration2 = 4.0
capture_framerate2 = 90
playback_fps2 = 25

mujoco.mj_resetData(model2, data2)
data2.qvel[:] = 0.0

# Initial conditions by ball
lower_pos = (0.0, 0.0, 0.24)
lower_vel = (0.0, 0.0, 0.0)
lower_spin = (0.0, 0.0, 5.0)

upper_pos = (0.0, 0.0, 0.50)
upper_vel = (0.0, 0.0, 0.0)
upper_spin = (0.0, 0.0, -5.0)

# Free-joint layout in qpos/qvel for two bodies:
# lower qpos[0:7], qvel[0:6]; upper qpos[7:14], qvel[6:12].
data2.qpos[0] = lower_pos[0]
data2.qpos[1] = lower_pos[1]
data2.qpos[2] = lower_pos[2]

data2.qvel[0] = lower_vel[0]
data2.qvel[1] = lower_vel[1]
data2.qvel[2] = lower_vel[2]
data2.qvel[3] = lower_spin[0]
data2.qvel[4] = lower_spin[1]
data2.qvel[5] = lower_spin[2]

data2.qpos[7] = upper_pos[0]
data2.qpos[8] = upper_pos[1]
data2.qpos[9] = upper_pos[2]

data2.qvel[6] = upper_vel[0]
data2.qvel[7] = upper_vel[1]
data2.qvel[8] = upper_vel[2]
data2.qvel[9] = upper_spin[0]
data2.qvel[10] = upper_spin[1]
data2.qvel[11] = upper_spin[2]

mujoco.mj_forward(model2, data2)
print(f"Lower ball IC -> pos={lower_pos}, vel={lower_vel}, spin={lower_spin}")
print(f"Upper ball IC -> pos={upper_pos}, vel={upper_vel}, spin={upper_spin}")

frames2 = []
with mujoco.Renderer(model2) as renderer2:
  while data2.time < duration2:
    mujoco.mj_step(model2, data2)
    if len(frames2) < data2.time * capture_framerate2:
      renderer2.update_scene(data2, "side")
      frames2.append(renderer2.render())

media.show_video(frames2, fps=playback_fps2)

Lower ball IC -> pos=(0.0, 0.0, 0.24), vel=(0.0, 0.0, 0.0), spin=(0.0, 0.0, 5.0)
Upper ball IC -> pos=(0.0, 0.0, 0.5), vel=(0.0, 0.0, 0.0), spin=(0.0, 0.0, -5.0)


This browser does not support the video tag.
